# Milestone 3 lab, notebook 4 of 8: The same person twice

## Where we are

| Notebook | What you do |
|---|---|
| 1. A price has a start and an end | products. See one forgotten date count every sale twice |
| 2. Six rows, wrong everywhere | stores. Find out why the weakest check is the best one here |
| 3. A gap is not a zero | stock_counts. Look for a row that is not there |
| **4. The same person twice** (you are here) | members. Find one person behind two keys, and keep their data private |
| 5. A file you do not control | supplier_prices. A supplier renames a column, and nothing fails |
| 6. Same shape, new meaning | supplier_prices. Every number is valid, and 100 times too high |
| 7. A contract for the supplier | Write down what a file must look like, and check it at the door |
| 8. Cards side by side | Compare the seven cards. Then see what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import keep, lab_copy, mask, on_lab, plant, prepare_week, production, show_card, sql

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 8 clean tables.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: Two emails to one person

**Marketing** sends an offer to every loyalty member who said yes to email. A member collects points on every receipt.

One customer of store `S03` is about to get every offer twice. Her points are split over two accounts, so she never reaches a reward. She has done nothing wrong. She signed up twice, months apart.

The table is `members`.

In [2]:
show_card("members", 1)

**From the check card `checks/members.md`, question 1. What does one row mean?**

One loyalty member: a person who signed up in the app. The key the app gives is `member_id`.

A row is about a **person**. Three of its columns are personal data: the name, the email and the phone number.

In [3]:
sql("""
    SELECT count(*)                  AS members,
           count(DISTINCT member_id) AS different_member_ids,
           count(DISTINCT email)     AS different_emails,
           count(*) - count(phone)   AS no_phone
    FROM clean.members
""")

,members,different_member_ids,different_emails,no_phone
0,308,308,308,118


308 members. Now the two checks everybody would write: the key is unique, and the email is unique.

In [4]:
by_id = sql("SELECT member_id, count(*) AS times FROM clean.members GROUP BY member_id HAVING count(*) > 1")
by_email = sql("SELECT email, count(*) AS times FROM clean.members GROUP BY email HAVING count(*) > 1")
print(len(by_id), "member ids are there twice")
print(len(by_email), "emails are there twice")

0 member ids are there twice
0 emails are there twice


Both pass. 308 ids, 308 emails, every one different. So every member is one person?

## Step 2: How the data is made

Question 3 of the card.

In [5]:
show_card("members", 3)

**From the check card `checks/members.md`, question 3. How is the data made? So what can go wrong?**

Customers type it themselves, on a phone, standing in a shop.

| Step | Made by | What can go wrong |
|---|---|---|
| A customer signs up | A person on a phone | A typing mistake in the email. Capital letters. A space at the end |
| The same customer signs up again | The same person, months later | A second `member_id` for one person |
| A customer skips a box | A person | A half-filled row |

A customer types this on a phone, standing in a shop. A phone likes to start a word with a capital letter. A thumb adds a space at the end.

To a database, `A@b.com` and `a@b.com` are two different texts. To the mail system they are one mailbox.

So before you ask "is it unique?", **clean the value**: remove the spaces around it, and make it lower case. In SQL that is `lower(trim(email))`.

In [6]:
sql("""
    SELECT count(DISTINCT email)              AS different_emails_as_typed,
           count(DISTINCT lower(trim(email))) AS different_emails_cleaned
    FROM clean.members
""")

,different_emails_as_typed,different_emails_cleaned
0,308,307


308 as typed. 307 after cleaning. One mailbox has two members.

Look at that pair. This is the one place in this notebook where a full email is printed, because you need to see the two texts. Every name and email in this project is invented.

In [7]:
sql("""
    SELECT member_id, email, home_store_id, joined_on
    FROM clean.members
    WHERE lower(trim(email)) IN (
        SELECT lower(trim(email)) FROM clean.members GROUP BY lower(trim(email)) HAVING count(*) > 1
    )
    ORDER BY member_id
""")

,member_id,email,home_store_id,joined_on
0,M-0117,priya.nair@example.com,S03,2026-06-21
1,M-0304,Priya.Nair@Example.com,S03,2026-08-26


`M-0117` joined in June. `M-0304` joined on 26 August, with the same email in other letter case, and the same home store.

This has a name: a **duplicate**. It is one real thing stored more than once under different keys.

Compare it with what you found in milestone 2. There, one receipt was in the file twice: the same thing, the same true key, two copies. A unique check on the right key finds that. A duplicate is harder. The keys are different, so no unique check on the key can ever see it. You must decide what makes two rows the **same person**, and clean that value first.

## Step 3: Write the check

In [8]:
ONE_PERSON_ONCE = """
    SELECT min(member_id) AS thing, max(joined_on) AS day,
           min(member_id) AS first_member_id, max(member_id) AS second_member_id,
           count(*) AS members_with_this_email
    FROM clean.members
    GROUP BY lower(trim(email))
    HAVING count(*) > 1
"""
yours = sql(ONE_PERSON_ONCE)
yours

,thing,day,first_member_id,second_member_id,members_with_this_email
0,M-0117,2026-08-26,M-0117,M-0304,2


In [9]:
keep("one_person_once", yours)

Your query:           1 bad rows
The production check: 1 bad rows  (One person is one member)
The same rows.


Try it on a fault of your own. In a private copy, give one member an email with capitals and spaces, and sign the same person up again.

In [10]:
lab_copy("clean.members")
plant("UPDATE lab.members SET email = '  ' || upper(email) || ' ' WHERE member_id = 'M-0010'")
plant("""
    INSERT INTO lab.members
    SELECT 'M-9999', full_name, lower(trim(email)), phone, home_store_id, joined_on, marketing_opt_in
    FROM lab.members WHERE member_id = 'M-0010'
""")
sql(on_lab(ONE_PERSON_ONCE))[["first_member_id", "second_member_id", "members_with_this_email"]]

lab.members is a fresh copy of clean.members: 308 rows.
Planted in the lab copy.
Planted in the lab copy.


,first_member_id,second_member_id,members_with_this_email
0,M-0010,M-9999,2
1,M-0117,M-0304,2


Two pairs now: the planted one and yours.

## Step 4: Look at what your check just printed

Go back to step 2. To show you the duplicate, a query printed a person's email on the screen.

Think about where a check's output goes in a real team. It is printed in a terminal. It is written into a log file that is kept for months. It is posted into a chat channel as an alert. In milestone 4 the bad rows are copied into another table.

A name, an email and a phone number are **personal data**: data about a person. The `members` table is protected. A log file is not. If the check copies the email into the log, the email has left the protected place.

So the production checks for this table follow one rule: **a bad row is named by its `member_id`, never by a name, an email or a phone number.** Whoever must fix it can look the member up, in the protected table.

Plant three more faults, and look at what each production check returns.

In [11]:
plant("UPDATE lab.members SET email = 'not an email' WHERE member_id = 'M-0005'")
plant("UPDATE lab.members SET full_name = NULL WHERE member_id = 'M-0006'")
plant("UPDATE lab.members SET home_store_id = 'S09' WHERE member_id = 'M-0007'")

personal = {"full_name", "email", "phone"}
for key in ("one_person_once", "email_has_the_right_shape", "member_cells_filled", "member_fits_the_real_world"):
    found = production(key, on_the_lab=True)
    leaked = personal & set(found.columns)
    print(f"{key:<28} {len(found)} bad rows   columns: {', '.join(found.columns)}")
    print(f"{'':<28} personal data in the result: {sorted(leaked) or 'none'}")

Planted in the lab copy.
Planted in the lab copy.
Planted in the lab copy.
one_person_once              2 bad rows   columns: thing, day, first_member_id, second_member_id, members_with_this_email
                             personal data in the result: none
email_has_the_right_shape    1 bad rows   columns: thing, day, problem
                             personal data in the result: none
member_cells_filled          1 bad rows   columns: thing, day, problem
                             personal data in the result: none


member_fits_the_real_world   1 bad rows   columns: thing, day, home_store_id, problem
                             personal data in the result: none


Four checks, each with bad rows, and not one name or email in any result.

When a person does need to see part of a value, show a part that cannot identify anyone. This is called masking.

In [12]:
pair = sql("SELECT member_id, email FROM clean.members WHERE member_id IN ('M-0117', 'M-0304') ORDER BY member_id")
pair["email"] = pair["email"].map(mask)
pair

,member_id,email
0,M-0117,p***@example.com
1,M-0304,p***@example.com


Both rows now show the same masked text. You can still see they belong together, and you cannot write to her.

## Step 5: What gets no check

In [13]:
show_card("members", 4)

**From the check card `checks/members.md`, question 4. Which of those would hurt most?**

| Rank | Worry | Why it is here |
|---|---|---|
| 1 | The same person twice | Two emails and two sets of points. Planted case 13 |
| 2 | An email that cannot be delivered | Marketing pays to send it, and nobody reads it |
| 3 | No answer to "may we email you?" | An email to someone who did not say yes |
| 4 | A member of a store that does not exist | Offers are sent by home store |
| 5 | An empty phone number | The app does not ask for it. **No check** |

One more worry is not about the rows. It is about the checks: a check prints its bad rows, and those rows hold a person's name and email.

118 members have no phone number. The app does not ask for one. So an empty phone hurts nobody, and it gets no check. You have seen this rule since milestone 1: a check with no reader to protect is deleted.

Compare that with "may we email you?". An empty answer there is serious: marketing must not write to someone who did not say yes. Same kind of fault, an empty cell. Different harm, so a different decision.

## What you now know

- A **duplicate** is one real thing stored more than once under different keys. Unique checks on the keys cannot see it.
- Clean a value before you compare it: `lower(trim(email))`. 308 emails as typed are 307 mailboxes.
- **Personal data** is data about a person. It must not travel into logs and check results. A bad row is named by its id.
- An empty phone gets no check. An empty answer to "may we email you?" does.

**Say it in an interview:** "The members table passed unique checks on the id and on the email, and one person was still in it twice, with her email in different letter case. I clean the value first, trim and lower, and then group. The check returns member ids only. A check's output ends up in logs and alerts, and personal data must not travel with it."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), the question "Unique?". Personal data in pipelines is part of [Security, Governance & Compliance](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/security-governance-and-compliance).

Next: [notebook 5](05_a_file_you_do_not_control.ipynb). Four tables were made inside Lotus. The last one is made by another company.